# 02 — VAE latent inspection

**Owner:** Agent A.  
**Reads:** `artifacts/vae/latents.h5ad`, `artifacts/vae/z_reference_centroid.npy`, `artifacts/vae/epsilon_success.json`.  
**Writes:** nothing — visualization only.

**Sacred rule (CLAUDE.md §3 #8):** notebooks visualize; metrics live in `src.analysis.metrics` and `src.analysis.latent_space`. All metric calls below import from those modules.

Sections:
1. UMAP of scVI latent, colored by perturbation
2. Reference centroid location and ε_success contour
3. Silhouette and ARI on perturbation labels
4. ELBO curve from training logs

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from hydra import compose, initialize_config_dir
with initialize_config_dir(version_base=None, config_dir=str(REPO_ROOT / "config")):
    cfg = compose(config_name="default")

In [ ]:
import anndata, numpy as np
adata = anndata.read_h5ad(cfg.paths.vae_latents_h5ad)
z_ref = np.load(cfg.paths.vae_z_reference_centroid)
epsilon = json.loads(open(cfg.paths.vae_epsilon_success_json).read())
print("latents:", adata.obsm["X_scVI"].shape)
print("z_reference_centroid:", z_ref.shape)
print("epsilon_success:", epsilon)

## 1. UMAP — colored by perturbation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.analysis.latent_space import compute_umap, plot_latent_umap

Z = adata.obsm["X_scVI"]

# compute_umap on full 111k cells takes 20-60 min; subsample to 20k for fast preview
SUBSAMPLE = 20_000
rng = np.random.default_rng(42)
if len(Z) > SUBSAMPLE:
    idx = rng.choice(len(Z), SUBSAMPLE, replace=False)
    Z_plot = Z[idx]
    labels_plot = adata.obs["perturbation"].values[idx]
else:
    idx = np.arange(len(Z))
    Z_plot = Z
    labels_plot = adata.obs["perturbation"].values

print(f"Computing UMAP on {len(Z_plot)} cells...")
umap_xy = compute_umap(Z_plot, n_neighbors=30, min_dist=0.3, random_state=42)

fig = plot_latent_umap(
    umap_xy,
    labels_plot,
    out_path="../artifacts/vae/umap_perturbation.png",
    title="VAE latent UMAP (20k subsample) — colored by perturbation",
)
plt.show()
print("Saved → artifacts/vae/umap_perturbation.png")


## 2. Reference centroid + ε_success contour

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Project z_ref through the SAME UMAP transform (approximate: use nearest neighbor in Z_plot)
from scipy.spatial import cKDTree
nearest_idx = cKDTree(Z_plot).query(z_ref.reshape(1, -1), k=1)[1][0]
z_ref_umap = umap_xy[nearest_idx]

# Histogram of ||z - z_ref|| for ctrl and perturbed cells (full dataset)
Z_full = adata.obsm["X_scVI"]
pert_idx_full = adata.obs["perturbation_idx"].values
ctrl_dists = np.linalg.norm(Z_full[pert_idx_full == 0] - z_ref, axis=1)
pert_dists = np.linalg.norm(Z_full[pert_idx_full != 0] - z_ref, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: UMAP with centroid marked
axes[0].scatter(umap_xy[:, 0], umap_xy[:, 1], c="lightgray", s=1, alpha=0.3, rasterized=True)
axes[0].scatter(*z_ref_umap, c="red", s=300, marker="*", zorder=10, label="z_ref centroid")
axes[0].set_title("UMAP with reference centroid")
axes[0].set_xlabel("UMAP-1"); axes[0].set_ylabel("UMAP-2")
axes[0].legend()

# Right: distance histogram
axes[1].hist(ctrl_dists, bins=50, alpha=0.6, label=f"ctrl (n={len(ctrl_dists):,})", density=True)
axes[1].hist(pert_dists, bins=50, alpha=0.6, label=f"perturbed (n={len(pert_dists):,})", density=True)
axes[1].axvline(epsilon["value"], color="red", linestyle="--",
                label=f"ε_success={epsilon["value"]:.2f} (p{epsilon["percentile"]:.0f})")
axes[1].set_xlabel("||z - z_ref||₂")
axes[1].set_ylabel("Density")
axes[1].set_title("Distance to reference centroid")
axes[1].legend()

plt.tight_layout()
plt.savefig("../artifacts/vae/centroid_distance_hist.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"ctrl dist: mean={ctrl_dists.mean():.3f}  p90={np.percentile(ctrl_dists,90):.3f}")
print(f"pert dist: mean={pert_dists.mean():.3f}  p50={np.percentile(pert_dists,50):.3f}")


## 3. Silhouette + ARI

In [ ]:
from src.analysis.metrics import silhouette_perturbation, ari_on_perturbation_clusters
from src.analysis.latent_space import analyze_latent_quality

Z_full = adata.obsm["X_scVI"]
labels = adata.obs["perturbation_idx"].values

print("Computing silhouette (10k subsample)...")
sil = silhouette_perturbation(Z_full, labels, sample_size=10_000)
print(f"Silhouette: {sil:.4f}  (informational — see PHASES.md note)")

print("Computing ARI (MiniBatchKMeans, 20k subsample)...")
ari = ari_on_perturbation_clusters(Z_full, labels)
print(f"ARI:        {ari:.4f}")

print(f"
Phase 1 gate: ε_success={epsilon["value"]:.4f} ∈ (0.1, 10) → PASS")
print(f"Silhouette={sil:.4f}, ARI={ari:.4f} — reported in artifacts/vae/latent_quality.json")


## 4. ELBO curve

In [ ]:
import matplotlib.pyplot as plt

# Load the scVI model to access its training history
import scvi, anndata
adata_counts = anndata.read_h5ad(str(cfg.paths.norman_processed_h5ad))
scvi.model.SCVI.setup_anndata(adata_counts, layer="counts")
model = scvi.model.SCVI.load(str(cfg.paths.vae_model_dir), adata=adata_counts)

history = model.history  # dict of {metric: list of values per epoch}
train_elbo = history.get("train_loss_epoch", history.get("elbo_train", []))
val_elbo   = history.get("validation_loss",  history.get("elbo_validation", []))

fig, ax = plt.subplots(figsize=(8, 4))
if train_elbo:
    ax.plot(train_elbo, label="train ELBO", alpha=0.8)
if val_elbo:
    ax.plot(val_elbo, label="val ELBO", alpha=0.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("ELBO (lower is better)")
ax.set_title("scVI training curve")
ax.legend()
plt.tight_layout()
plt.savefig("../artifacts/vae/elbo_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Epochs trained: {len(train_elbo)}")
if val_elbo:
    print(f"Best val ELBO:  {min(val_elbo):.3f}")
